# Limpeza e Preparação dos Dados — ENEM 2023

**Projeto:** EduDataBR  
**Sprint:** 4 — Limpeza & Análise Exploratória  
**Entrada:** `data/interim/microdados_enem_2023_reduzido.csv` (18 colunas, 3.933.955 registros)  
**Saídas:**
- `data/processed/enem_2023_populacional.parquet` — todos os inscritos (estudos de perfil e evasão)
- `data/processed/enem_2023_analitico.parquet` — presentes nas 4 provas (estudos de desempenho)

Este notebook aplica as regras definidas em [`docs/03_data_quality.md`](../docs/03_data_quality.md) e as decisões D1–D6 da Sprint 4:

| Decisão | O que foi definido |
|---|---|
| D1 | Notas zero na Redação são **mantidas** e marcadas com `fl_redacao_zero` |
| D2 | Notas zero nas objetivas são **mantidas** e marcadas com `fl_objetiva_zero` |
| D4 | O CSV intermediário é a fonte; o PostgreSQL é usado só para **validação cruzada** |

> Nenhuma nota ausente é imputada: os nulos significam que o participante faltou ou foi eliminado (ausência estrutural, MNAR).

## 1. Preparação do ambiente

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

INTERIM_PATH = Path("../data/interim/microdados_enem_2023_reduzido.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

POP_PATH = PROCESSED_DIR / "enem_2023_populacional.parquet"
ANA_PATH = PROCESSED_DIR / "enem_2023_analitico.parquet"

print("pandas", pd.__version__)

pandas 3.0.3


## 2. Carregamento do CSV intermediário

`NU_INSCRICAO` é lido como texto para não perder precisão (o número tem 12 dígitos). O tempo de leitura fica registrado para a comparação com o Parquet na seção 9.

In [2]:
inicio = time.perf_counter()
df = pd.read_csv(INTERIM_PATH, sep=";", dtype={"NU_INSCRICAO": str})
tempo_csv = time.perf_counter() - inicio

memoria_inicial_mb = df.memory_usage(deep=True).sum() / 1024**2

print(f"Registros: {len(df):,}")
print(f"Colunas:   {df.shape[1]}")
print(f"Tempo de leitura do CSV: {tempo_csv:.1f} s")
print(f"Memória com tipos padrão: {memoria_inicial_mb:,.0f} MB")

Registros: 3,933,955
Colunas:   18
Tempo de leitura do CSV: 5.7 s
Memória com tipos padrão: 604 MB


In [3]:
df.head()

,NU_INSCRICAO,NU_ANO,TP_FAIXA_ETARIA,TP_SEXO,TP_COR_RACA,SG_UF_PROVA,TP_ESCOLA,TP_PRESENCA_CN,TP_PRESENCA_CH,TP_PRESENCA_LC,TP_PRESENCA_MT,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,NU_NOTA_REDACAO,Q006,Q025
0,210059085136,2023,14,M,1,DF,1,0,0,0,0,NaN,NaN,NaN,NaN,NaN,F,B
1,210059527735,2023,12,M,1,DF,1,0,0,0,0,NaN,NaN,NaN,NaN,NaN,H,B
2,210061103945,2023,6,F,1,RS,1,1,1,1,1,502.00,498.90,475.60,363.20,700.00,C,B
3,210060214087,2023,2,F,3,CE,2,1,1,1,1,459.00,508.50,507.20,466.70,880.00,C,B
4,210059980948,2023,3,F,3,CE,2,1,1,1,1,402.50,379.20,446.90,338.30,560.00,B,A


## 3. Checagem do schema

Garante que o arquivo tem exatamente as 18 colunas documentadas no dicionário de dados, na ordem esperada. Se o CSV intermediário for regerado com outras colunas, o notebook para aqui.

In [4]:
COLUNAS_ESPERADAS = [
    "NU_INSCRICAO", "NU_ANO", "TP_FAIXA_ETARIA", "TP_SEXO", "TP_COR_RACA",
    "SG_UF_PROVA", "TP_ESCOLA",
    "TP_PRESENCA_CN", "TP_PRESENCA_CH", "TP_PRESENCA_LC", "TP_PRESENCA_MT",
    "NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT", "NU_NOTA_REDACAO",
    "Q006", "Q025",
]

assert list(df.columns) == COLUNAS_ESPERADAS, "Schema diferente do documentado"
print("Schema OK: 18 colunas na ordem esperada.")

Schema OK: 18 colunas na ordem esperada.


## 4. Validações de qualidade

As mesmas regras auditadas em `docs/03_data_quality.md`, agora como `assert`. Se alguma regra falhar, o notebook interrompe a execução em vez de gerar uma base inconsistente.

| Regra | Verificação |
|---|---|
| Unicidade | `NU_INSCRICAO` sem duplicatas |
| Completude | colunas cadastrais e socioeconômicas sem nulos |
| Validade | domínios categóricos dentro do esperado |
| Consistência | presente ⇔ nota preenchida (a Redação acompanha a presença em LC, no Dia 1) |
| Acurácia | notas entre 0 e 1000 |

In [5]:
PRESENCAS = ["TP_PRESENCA_CN", "TP_PRESENCA_CH", "TP_PRESENCA_LC", "TP_PRESENCA_MT"]
NOTAS = ["NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT", "NU_NOTA_REDACAO"]
NOTAS_OBJETIVAS = ["NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT"]

# Unicidade
assert df["NU_INSCRICAO"].is_unique

# Completude
colunas_sem_nulo = [c for c in COLUNAS_ESPERADAS if c not in NOTAS]
assert df[colunas_sem_nulo].notna().all().all()

# Validade
assert set(df["NU_ANO"].unique()) == {2023}
assert set(df["TP_SEXO"].unique()) <= {"M", "F"}
assert set(df["TP_COR_RACA"].unique()) <= {0, 1, 2, 3, 4, 5}
assert df["TP_FAIXA_ETARIA"].between(1, 20).all()
assert set(df["TP_ESCOLA"].unique()) <= {1, 2, 3}
assert df["SG_UF_PROVA"].nunique() == 27
assert set(df["Q006"].unique()) <= set("ABCDEFGHIJKLMNOPQ")
assert set(df["Q025"].unique()) <= {"A", "B"}
for col in PRESENCAS:
    assert set(df[col].unique()) <= {0, 1, 2}

# Consistência presença x nota
pares = [
    ("TP_PRESENCA_CN", "NU_NOTA_CN"),
    ("TP_PRESENCA_CH", "NU_NOTA_CH"),
    ("TP_PRESENCA_LC", "NU_NOTA_LC"),
    ("TP_PRESENCA_MT", "NU_NOTA_MT"),
    ("TP_PRESENCA_LC", "NU_NOTA_REDACAO"),
]
for presenca, nota in pares:
    assert ((df[presenca] == 1) == df[nota].notna()).all(), f"Inconsistência em {nota}"

# Acurácia
for nota in NOTAS:
    assert df[nota].dropna().between(0, 1000).all()

print("Todas as validações de qualidade passaram.")

Todas as validações de qualidade passaram.


## 5. Otimização de tipos

Com os tipos padrão, o Pandas usa `int64`/`float64` e texto genérico. Como os valores cabem em tipos menores, a conversão reduz bastante a memória:

| Tipo novo | Colunas | Motivo |
|---|---|---|
| `int8` | códigos (`TP_*`) | valores de 0 a 20 |
| `int16` | `NU_ANO` | valor 2023 |
| `float32` | notas | precisão de 1 casa decimal é suficiente para armazenar |
| `category` | `TP_SEXO`, `SG_UF_PROVA`, `Q025` | poucos valores distintos repetidos milhões de vezes |
| `category` ordenada | `Q006` | as faixas de renda têm ordem (A < B < … < Q) |

> Os **cálculos** de média feitos mais adiante convertem as notas para `float64` antes de agregar, para não acumular erro de arredondamento do `float32` em milhões de linhas.

In [6]:
CODIGOS_INT8 = ["TP_FAIXA_ETARIA", "TP_COR_RACA", "TP_ESCOLA"] + PRESENCAS

df[CODIGOS_INT8] = df[CODIGOS_INT8].astype("int8")
df["NU_ANO"] = df["NU_ANO"].astype("int16")
df[NOTAS] = df[NOTAS].astype("float32")

df["TP_SEXO"] = df["TP_SEXO"].astype("category")
df["SG_UF_PROVA"] = df["SG_UF_PROVA"].astype("category")
df["Q025"] = df["Q025"].astype("category")
df["Q006"] = pd.Categorical(df["Q006"], categories=list("ABCDEFGHIJKLMNOPQ"), ordered=True)

memoria_tipada_mb = df.memory_usage(deep=True).sum() / 1024**2
reducao = 1 - memoria_tipada_mb / memoria_inicial_mb

print(f"Memória antes: {memoria_inicial_mb:,.0f} MB")
print(f"Memória depois: {memoria_tipada_mb:,.0f} MB")
print(f"Redução: {reducao:.1%}")

Memória antes: 604 MB
Memória depois: 199 MB
Redução: 67.1%


In [7]:
df.dtypes

NU_INSCRICAO            str
NU_ANO                int16
TP_FAIXA_ETARIA        int8
TP_SEXO            category
TP_COR_RACA            int8
SG_UF_PROVA        category
TP_ESCOLA              int8
TP_PRESENCA_CN         int8
TP_PRESENCA_CH         int8
TP_PRESENCA_LC         int8
TP_PRESENCA_MT         int8
NU_NOTA_CN          float32
NU_NOTA_CH          float32
NU_NOTA_LC          float32
NU_NOTA_MT          float32
NU_NOTA_REDACAO     float32
Q006               category
Q025               category
dtype: object

## 6. Colunas derivadas para a EDA

Apenas colunas necessárias para a análise exploratória. As features de modelagem (`nu_nota_media_geral`, `classe_renda`, `in_vulnerabilidade`) ficam para a Sprint 5.

### 6.1 Flags de presença, evasão e notas zero

| Coluna | Regra |
|---|---|
| `fl_presente_total` | presente (`1`) nas 4 provas |
| `fl_evasao_dia2` | presente no Dia 1 (`TP_PRESENCA_LC = 1`) e **faltou** no Dia 2 (`TP_PRESENCA_MT = 0`). Eliminados no Dia 2 (`2`) não contam como evasão. É a mesma regra das queries SQL 10 e da view `vw_resumo_por_uf`, o que permite comparar os resultados |
| `fl_redacao_zero` | nota da Redação igual a 0 (decisão D1) |
| `fl_objetiva_zero` | pelo menos uma prova objetiva com nota 0 (decisão D2) |

In [8]:
df["fl_presente_total"] = (df[PRESENCAS] == 1).all(axis=1)
df["fl_evasao_dia2"] = (df["TP_PRESENCA_LC"] == 1) & (df["TP_PRESENCA_MT"] == 0)
df["fl_redacao_zero"] = df["NU_NOTA_REDACAO"].eq(0)
df["fl_objetiva_zero"] = df[NOTAS_OBJETIVAS].eq(0).any(axis=1)

df[["fl_presente_total", "fl_evasao_dia2", "fl_redacao_zero", "fl_objetiva_zero"]].sum().to_frame("qtd_true")

,qtd_true
fl_presente_total,2678264
fl_evasao_dia2,142219
fl_redacao_zero,117829
fl_objetiva_zero,22466


### 6.2 Macrorregião

Necessária para a PQ5 (assimetrias regionais). Mapeamento oficial do IBGE das 27 UFs nas 5 regiões.

In [9]:
REGIOES = {
    "Norte": ["AC", "AP", "AM", "PA", "RO", "RR", "TO"],
    "Nordeste": ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"],
    "Centro-Oeste": ["DF", "GO", "MT", "MS"],
    "Sudeste": ["ES", "MG", "RJ", "SP"],
    "Sul": ["PR", "RS", "SC"],
}
UF_PARA_REGIAO = {uf: regiao for regiao, ufs in REGIOES.items() for uf in ufs}

assert len(UF_PARA_REGIAO) == 27
df["regiao"] = df["SG_UF_PROVA"].astype(str).map(UF_PARA_REGIAO).astype("category")
assert df["regiao"].notna().all()

df["regiao"].value_counts()

regiao
Nordeste        1423999
Sudeste         1305362
Norte            453454
Sul              417688
Centro-Oeste     333452
Name: count, dtype: int64

### 6.3 Rótulos legíveis

Colunas descritivas para os gráficos e tabelas. Os códigos originais continuam na base.

In [10]:
ROTULO_ESCOLA = {1: "Não respondeu", 2: "Pública", 3: "Privada"}
ROTULO_COR_RACA = {0: "Não declarado", 1: "Branca", 2: "Preta", 3: "Parda", 4: "Amarela", 5: "Indígena"}
ROTULO_SEXO = {"F": "Feminino", "M": "Masculino"}
ROTULO_INTERNET = {"A": "Sem internet", "B": "Com internet"}
ROTULO_RENDA = {
    "A": "Nenhuma renda",
    "B": "Até R$ 1.320",
    "C": "R$ 1.320–1.980",
    "D": "R$ 1.980–2.640",
    "E": "R$ 2.640–3.300",
    "F": "R$ 3.300–3.960",
    "G": "R$ 3.960–5.280",
    "H": "R$ 5.280–6.600",
    "I": "R$ 6.600–7.920",
    "J": "R$ 7.920–9.240",
    "K": "R$ 9.240–10.560",
    "L": "R$ 10.560–11.880",
    "M": "R$ 11.880–13.200",
    "N": "R$ 13.200–15.840",
    "O": "R$ 15.840–19.800",
    "P": "R$ 19.800–26.400",
    "Q": "Acima de R$ 26.400",
}


def rotular(serie, mapa, ordenado=False):
    valores = serie.astype(object).map(mapa)
    return pd.Categorical(valores, categories=list(mapa.values()), ordered=ordenado)


df["escola_desc"] = rotular(df["TP_ESCOLA"], ROTULO_ESCOLA)
df["cor_raca_desc"] = rotular(df["TP_COR_RACA"], ROTULO_COR_RACA)
df["sexo_desc"] = rotular(df["TP_SEXO"], ROTULO_SEXO)
df["internet_desc"] = rotular(df["Q025"], ROTULO_INTERNET)
df["renda_desc"] = rotular(df["Q006"], ROTULO_RENDA, ordenado=True)

for col in ["escola_desc", "cor_raca_desc", "sexo_desc", "internet_desc", "renda_desc"]:
    assert df[col].notna().all(), f"Rótulo faltando em {col}"

df[["TP_ESCOLA", "escola_desc", "Q006", "renda_desc", "Q025", "internet_desc"]].drop_duplicates().head(10)

,TP_ESCOLA,escola_desc,Q006,renda_desc,Q025,internet_desc
0,1,Não respondeu,F,R$ 3.300–3.960,B,Com internet
1,1,Não respondeu,H,R$ 5.280–6.600,B,Com internet
2,1,Não respondeu,C,R$ 1.320–1.980,B,Com internet
3,2,Pública,C,R$ 1.320–1.980,B,Com internet
4,2,Pública,B,Até R$ 1.320,A,Sem internet
7,1,Não respondeu,A,Nenhuma renda,B,Com internet
10,1,Não respondeu,B,Até R$ 1.320,B,Com internet
11,2,Pública,D,R$ 1.980–2.640,B,Com internet
12,2,Pública,B,Até R$ 1.320,B,Com internet
17,1,Não respondeu,B,Até R$ 1.320,A,Sem internet


## 7. Investigação das notas zero (decisão D2)

Antes de seguir com a decisão de manter as notas zero, vale checar se elas se concentram em algum perfil. A comparação é feita **entre os presentes nas 4 provas**: se a proporção de zeros num grupo for muito diferente da proporção geral, isso precisa ser levado em conta na EDA.

In [11]:
presentes = df[df["fl_presente_total"]]

def taxa_por_grupo(base, grupo):
    return (
        base.groupby(grupo, observed=True)[["fl_redacao_zero", "fl_objetiva_zero"]]
        .mean()
        .mul(100)
        .rename(columns={"fl_redacao_zero": "% redação zero", "fl_objetiva_zero": "% objetiva zero"})
    )

geral = presentes[["fl_redacao_zero", "fl_objetiva_zero"]].mean().mul(100)
print(f"Geral entre presentes -> redação zero: {geral['fl_redacao_zero']:.2f}% | objetiva zero: {geral['fl_objetiva_zero']:.2f}%")

taxa_por_grupo(presentes, "escola_desc")

Geral entre presentes -> redação zero: 3.48% | objetiva zero: 0.73%


,% redação zero,% objetiva zero
escola_desc,,
Não respondeu,3.09,0.78
Pública,4.96,0.76
Privada,0.74,0.24


In [12]:
taxa_por_grupo(presentes, "renda_desc")

,% redação zero,% objetiva zero
renda_desc,,
Nenhuma renda,6.59,0.97
Até R$ 1.320,5.20,0.85
R$ 1.320–1.980,3.53,0.81
R$ 1.980–2.640,3.02,0.74
R$ 2.640–3.300,2.54,0.63
R$ 3.300–3.960,2.27,0.62
R$ 3.960–5.280,1.96,0.56
R$ 5.280–6.600,1.69,0.56
R$ 6.600–7.920,1.50,0.51


## 8. Particionamento e exportação em Parquet

Seguindo a Regra 1 de `docs/03_data_quality.md`:

- **Base populacional:** todos os 3.933.955 inscritos.
- **Base analítica:** apenas `fl_presente_total = True` (esperado: 2.678.264, valor corrigido na revisão da Sprint 4).

In [13]:
df_pop = df
df_ana = df[df["fl_presente_total"]].reset_index(drop=True)

assert len(df_pop) == 3_933_955
assert len(df_ana) == 2_678_264
assert df_ana[NOTAS].notna().all().all(), "A base analítica não deveria ter notas nulas"

print(f"Base populacional: {len(df_pop):,} registros")
print(f"Base analítica:    {len(df_ana):,} registros")

Base populacional: 3,933,955 registros
Base analítica:    2,678,264 registros


In [14]:
df_pop.to_parquet(POP_PATH, index=False)
df_ana.to_parquet(ANA_PATH, index=False)

tamanho_csv_mb = INTERIM_PATH.stat().st_size / 1024**2
tamanho_pop_mb = POP_PATH.stat().st_size / 1024**2
tamanho_ana_mb = ANA_PATH.stat().st_size / 1024**2

pd.DataFrame(
    {
        "arquivo": [INTERIM_PATH.name, POP_PATH.name, ANA_PATH.name],
        "registros": [len(df_pop), len(df_pop), len(df_ana)],
        "colunas": [18, df_pop.shape[1], df_ana.shape[1]],
        "tamanho_mb": [tamanho_csv_mb, tamanho_pop_mb, tamanho_ana_mb],
    }
)

,arquivo,registros,colunas,tamanho_mb
0,microdados_enem_2023_reduzido.csv,3933955,18,242.75
1,enem_2023_populacional.parquet,3933955,28,65.76
2,enem_2023_analitico.parquet,2678264,28,45.63


## 9. Leitura do Parquet: tempo e preservação dos tipos

Relê a base populacional para medir o tempo de leitura e confirmar que os tipos otimizados (categorias, `int8`, `float32`) foram preservados no arquivo.

In [15]:
inicio = time.perf_counter()
df_pop_lido = pd.read_parquet(POP_PATH)
tempo_parquet = time.perf_counter() - inicio

assert df_pop_lido.shape == df_pop.shape
assert (df_pop_lido.dtypes == df_pop.dtypes).all(), "Algum tipo não foi preservado"

print(f"Leitura do CSV (18 colunas):              {tempo_csv:.1f} s")
print(f"Leitura do Parquet ({df_pop.shape[1]} colunas, já tipado): {tempo_parquet:.1f} s")
print(f"Tipos preservados: sim")

del df_pop_lido

Leitura do CSV (18 colunas):              5.7 s
Leitura do Parquet (28 colunas, já tipado): 0.5 s
Tipos preservados: sim


## 10. Validação cruzada com o PostgreSQL

Os números abaixo são calculados no Pandas a partir da base analítica, com as mesmas regras de duas queries SQL do projeto. Para validar, rode as queries no pgAdmin e compare:

| Tabela abaixo | Query no pgAdmin |
|---|---|
| 10.1 Médias por faixa de renda | [`sql/queries/04_desempenho_por_renda.sql`](../sql/queries/04_desempenho_por_renda.sql) |
| 10.2 Ranking das UFs | [`sql/queries/06_ranking_estados_window_functions.sql`](../sql/queries/06_ranking_estados_window_functions.sql) |

As médias são calculadas em `float64`. Diferenças de 0,01 podem aparecer pelo arredondamento final; diferenças maiores indicam algum problema.

In [16]:
notas64 = df_ana[NOTAS].astype("float64")

validacao_renda = (
    notas64.assign(faixa_renda=df_ana["Q006"])
    .groupby("faixa_renda", observed=True)
    .agg(
        total_participantes=("NU_NOTA_MT", "size"),
        media_matematica=("NU_NOTA_MT", "mean"),
        media_redacao=("NU_NOTA_REDACAO", "mean"),
        media_cn=("NU_NOTA_CN", "mean"),
        media_ch=("NU_NOTA_CH", "mean"),
        media_lc=("NU_NOTA_LC", "mean"),
    )
    .round(2)
)
validacao_renda

,total_participantes,media_matematica,media_redacao,media_cn,media_ch,media_lc
faixa_renda,,,,,,
A,155599,454.86,530.48,453.06,478.22,475.76
B,779111,476.80,568.28,464.26,493.22,491.35
C,414445,510.45,601.07,483.30,516.06,512.24
D,295876,530.85,622.59,494.64,528.22,522.99
E,209516,549.77,644.07,504.64,537.92,531.79
F,127207,568.34,661.60,513.64,546.40,538.68
G,201713,584.47,681.26,523.09,555.26,545.91
H,110432,602.32,700.62,532.77,563.77,552.89
I,69432,613.64,714.85,539.54,569.36,557.54


In [17]:
validacao_uf = (
    notas64.assign(
        sg_uf_prova=df_ana["SG_UF_PROVA"].astype(str),
        media_geral=notas64.mean(axis=1),
    )
    .groupby("sg_uf_prova")
    .agg(
        total_participantes=("NU_NOTA_MT", "size"),
        media_matematica=("NU_NOTA_MT", "mean"),
        media_redacao=("NU_NOTA_REDACAO", "mean"),
        media_geral=("media_geral", "mean"),
    )
    .round(2)
    .sort_values("media_geral", ascending=False)
)
validacao_uf["ranking_nacional"] = validacao_uf["media_geral"].rank(ascending=False, method="min").astype(int)
validacao_uf

,total_participantes,media_matematica,media_redacao,media_geral,ranking_nacional
sg_uf_prova,,,,,
MG,250015,567.59,662.39,565.14,1
SP,409480,572.39,645.11,564.92,2
SC,62700,562.02,642.22,559.62,3
DF,50267,554.83,640.19,558.87,4
RS,110735,558.02,639.82,557.56,5
ES,49235,556.14,648.35,556.55,6
RJ,184502,552.13,644.40,555.99,7
PR,114776,553.30,616.25,550.06,8
GO,96351,532.90,641.93,542.79,9
